In [2]:
# Building Chatbot with multiple tools using langgraph

# aim 
# create a chatbot with tool capabilities from arxiv , wikipedia search and some functions

In [3]:
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [5]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=2,doc_content_chars_max=500)

arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)

arxiv


In [6]:
arxiv.invoke("what is attention is all you need")

'Published: 2021-05-06\nTitle: Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet\nAuthors: Luke Melas-Kyriazi\nSummary: The strong performance of vision transformers on image classification and other vision tasks is often attributed to the design of their multi-head attention layers. However, the extent to which attention is responsible for this strong performance remains unclear. In this short report, we ask: is the attention layer even necessary? Specifi'

In [7]:
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=2,doc_content_chars_max=500)

wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
print(wiki.name)

wikipedia


In [16]:
import time

try:
    result = wiki.invoke("machine learning")
    print(result)
except Exception as e:
    print(f"Error: {e}")
    print("Wikipedia API is temporarily unavailable. Trying alternative query...")
    time.sleep(2)
    try:
        result = wiki.invoke("artificial intelligence")
        print(result)
    except Exception as e2:
        print(f"Still having issues: {e2}")
        print("Using a simpler query...")
        try:
            result = wiki.invoke("AI")
            print(result)
        except Exception as e3:
            print(f"Wikipedia API is not responding properly: {e3}")

Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
Statistics and mathematical optimisation me


In [15]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

python-dotenv could not parse statement starting at line 8


In [17]:
# Tavily search tool
from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults()

C:\Users\hp\AppData\Local\Temp\ipykernel_22976\231603297.py:4: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily = TavilySearchResults()


In [19]:
tavily.invoke("Provide me the recent AI news for may 12 2026")

[{'title': 'Artificial Intelligence - AI Update, May 8, 2026: AI News and Views From the Past Week',
  'url': 'https://www.marketingprofs.com/opinions/2026/54655/ai-update-may-8-2026-ai-news-and-views-from-the-past-week',
  'content': 'Importance for marketers: The collapse of a major AI-search integration partnership highlights how unsettled AI platform partnerships and monetization strategies remain. Social platforms, search experiences, and conversational discovery ecosystems are still evolving rapidly, with uncertain winners and business models. [...] Importance for marketers: The AI industry is increasingly acknowledging that deployment services, integration expertise, and operational implementation may become just as strategically important as model development itself. That shift could reshape enterprise AI buying decisions, agency offerings, consulting markets, and martech partnerships. [...] OpenAI launches B2B Signals to track enterprise AI adoption patterns. OpenAI has introd

In [20]:
# Combine all the tools in the list

tools = [arxiv,wiki,tavily]



In [21]:
# Initialize my LLM model

from langchain_groq import ChatGroq

llm = ChatGroq(model="qwen/qwen3-32b")

llm_with_tools = llm.bind_tools(tools)

In [23]:
from langchain_core.messages import HumanMessage,AIMessage


result = llm_with_tools.invoke([HumanMessage(content="what is the recent ai news")])

In [25]:
result

AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about recent AI news. I need to figure out which tool to use here. Let me check the available tools.\n\nFirst, there\'s the arxiv function, which is for scientific papers in fields like physics and computer science. But the user is asking for news, not academic papers. Maybe not the best fit.\n\nNext, Wikipedia. That\'s for general knowledge and historical information. While AI topics might be covered there, it might not have the latest news. Wikipedia updates can be a bit slow for current events.\n\nThen there\'s tavily_search_results_json. The description says it\'s optimized for current events with comprehensive and trusted results. That sounds perfect for recent news. The user probably wants up-to-date information on AI developments.\n\nSo I should use the tavily_search_results_json function with the query "recent AI news". That should fetch the latest updates they\'re looking for.\n', 'tool_cal

In [26]:
result.tool_calls

[{'name': 'tavily_search_results_json',
  'args': {'query': 'recent AI news'},
  'id': '385v52y8n',
  'type': 'tool_call'}]